# 第15周 Day1 — LnkChatBI NL→SQL 组装链路：四个可执行概念实验

**md 是阅读材料（链路九级流水 + L1-L5 验收口径），本 notebook 是概念验证**：把精读发现的四个关键结构做成可执行实验——

| 实验 | 验证的概念 | 对应代码事实 |
|---|---|---|
| ① SSE 契约不变量校验器 | 事件顺序契约可机器校验（不是文档约定） | `streaming/events.py` + `ChartAnswer.vue` 13 类事件 |
| ② 权限下推蒙特卡洛 | 行权限是**概率性执行**（LLM 自报表 + fallback 双通路，静默丢失） | `check_sql` tables 自报 + `get_row_permission_filters` 空表返回 |
| ③ 表注册对账提取对照 | 确定性加固路径：对着表注册扫 SQL，召回恒 100% | `check_sql_read` 用 sqlglot AST 的思路对照 |
| ④ L1-L3 验收判定器 v0.1 | D6 Demo 验收口径可执行化（种子数据实测素材） | PT-W4-D6 判分表 × mallcre `bi_d_position` |

约束：numpy/matplotlib/标准库；无联网、无 LLM 调用（LLM 行为用参数化概率模型代替，参数在单元格内可调）。

In [ ]:
# 字体配置：TOOLS.md 唯一标准（所有 notebook 必须用这个）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)
import numpy as np
import os
OUT = "/root/learning-notebooks/第15周"
os.makedirs(OUT, exist_ok=True)

## 实验①：SSE 事件契约不变量校验器

openspec `chat-streaming-contract` 规定：共享序列化、错误统一形状、**finish 只在编排层终态后发**、前端共享适配器。
把契约写成 7 条不变量，对 4 条轨迹（A 正常带选库 / B 正常已定库 / C 中途出错 / D 故意破坏）做机器校验：

| # | 不变量 |
|---|---|
| I1 | 首事件必须是 `id` |
| I2 | `question` 在 `id` 之后、任何生成类事件之前 |
| I3 | `sql`（终版）在 `sql-data` 之前 |
| I4 | `sql-data` 在 `chart-result`/`chart` 之前 |
| I5 | `finish` 只能是最后一条事件 |
| I6 | `error` 是终止事件（其后不得再有事件） |
| I7 | 流式事件（`*-result`）不得出现在 `finish`/`error` 之后 |

In [ ]:
def validate_sse(events):
    """对一条 SSE 事件序列校验 7 条不变量，返回 (ok, violations)。"""
    v = []
    types = [e["type"] for e in events]
    def idx(t):
        try:
            return types.index(t)
        except ValueError:
            return None
    if not types or types[0] != "id":
        v.append("I1: 首事件不是 id")
    iq, iid = idx("question"), idx("id")
    gen_first = min([i for i, t in enumerate(types) if t in ("sql-result", "sql", "chart-result")], default=10**9)
    if iq is None or (iid is not None and iq < iid) or iq > gen_first:
        v.append("I2: question 位置非法")
    isql, idata = idx("sql"), idx("sql-data")
    if isql is not None and idata is not None and isql > idata:
        v.append("I3: sql 终版晚于 sql-data")
    ichart, icr = idx("chart"), idx("chart-result")
    if idata is not None and ((icr is not None and icr < idata) or (ichart is not None and ichart < idata)):
        v.append("I4: chart 类事件早于 sql-data")
    ifinish, ierr = idx("finish"), idx("error")
    if ifinish is not None and ifinish != len(types) - 1:
        v.append("I5: finish 不是最后一条")
    if ierr is not None and ierr != len(types) - 1:
        v.append("I6: error 后仍有事件")
    term = min([i for i in (ifinish, ierr) if i is not None], default=None)
    if term is not None:
        for i, t in enumerate(types[term + 1:], start=term + 1):
            if t.endswith("-result"):
                v.append("I7: 终止事件后出现流式事件 @%d" % i)
                break
    return (len(v) == 0), v

# 四条轨迹（事件名与顺序严格来自 run_task 代码事实）
traj_A = [  # 正常：未定库 → LLM 选库
    {"type": "id"}, {"type": "question"},
    {"type": "datasource-result"}, {"type": "datasource"},
    {"type": "sql-result"}, {"type": "info"}, {"type": "brief"},
    {"type": "sql"}, {"type": "sql-data"},
    {"type": "chart-result"}, {"type": "chart"}, {"type": "finish"},
]
traj_B = [  # 正常：已定库（无 datasource 段）
    {"type": "id"}, {"type": "question"},
    {"type": "sql-result"}, {"type": "info"},
    {"type": "sql"}, {"type": "sql-data"},
    {"type": "chart-result"}, {"type": "chart"}, {"type": "finish"},
]
traj_C = [  # 出错：SQL 生成中失败 → error 终止
    {"type": "id"}, {"type": "question"},
    {"type": "sql-result"}, {"type": "error"},
]
traj_D = [  # 故意破坏：sql-data 抢在 sql 前 + error 后还发流式事件
    {"type": "id"}, {"type": "question"},
    {"type": "sql-data"}, {"type": "sql"},
    {"type": "error"}, {"type": "chart-result"},
]

print("%-8s%-8s%-8s%s" % ("轨迹", "事件数", "判定", "违规项"))
for name, t in [("A", traj_A), ("B", traj_B), ("C", traj_C), ("D", traj_D)]:
    ok, v = validate_sse(t)
    print("%-8s%-8d%-8s%s" % (name, len(t), "PASS" if ok else "FAIL", "; ".join(v) if v else "—"))

# 可视化：三条合法轨迹的事件时间线
fig, axes = plt.subplots(3, 1, figsize=(10, 6.5))
for ax, (name, t) in zip(axes, [("A 选库流", traj_A), ("B 定库流", traj_B), ("C 错误流", traj_C)]):
    labels = [e["type"] for e in t]
    colors = ["#d62728" if l == "error" else ("#2ca02c" if l == "finish" else "#1f77b4") for l in labels]
    ax.scatter(range(len(t)), [0] * len(t), marker=">", s=90, c=colors, zorder=3)
    for x, l in enumerate(labels):
        ax.annotate(l, (x, 0), xytext=(x, 0.18 + (0.14 if x % 2 else 0)), ha="center", fontsize=8)
    ax.set_yticks([]); ax.set_ylim(-0.5, 0.65); ax.set_title("轨迹" + name, fontsize=10)
    ax.set_xticks(range(len(t)))
fig.suptitle("LnkChatBI SSE 事件契约：三条合法轨迹（run_task 代码实测顺序）", fontsize=12)
plt.tight_layout()
plt.savefig(OUT + "/w15d1_sse_contract.png", dpi=110, bbox_inches="tight")
plt.show()
print("结论：契约 = 可机器校验的不变量集合，而非文档里的时序图。D6 Demo 录制时可复用该校验器做回归断言。")

## 实验②：权限下推蒙特卡洛——「概率性执行」到底多概率

代码事实（`llm.py` + `permission.py`）：行权限改写的输入 `tables` 来自 **LLM JSON 自报**；LLM 漏报的表静默失去行过滤；
LLM 返回裸 SQL 走 fallback 时 `tables=None` → **整条查询跳过权限改写**（fallback 安全判定只查单语句+只读，不查权限）。

概率模型（参数可调）：主表自报召回 `p_main`，每个 join 表 `p_join`，非 JSON fallback 概率 `p_fb`；
一次查询「存在权限缺口」⟺ fallback 发生 或 任一表被漏报。另对比确定性闸门（AST 只读）恒为 0。

In [ ]:
rng = np.random.default_rng(42)
N = 200_000  # 每组蒙特卡洛次数
p_main, p_join, p_fb = 0.99, 0.93, 0.02   # 单次自报召回 / fallback 概率（保守取值，可调）

def sim_hole_prob(J, n=N, p_main=p_main, p_join=p_join, p_fb=p_fb):
    """J = 该查询涉及的受保护表数（含主表）。返回存在权限缺口的频率。"""
    recalls = np.array([p_main] + [p_join] * (J - 1))
    fb = rng.random(n) < p_fb                      # 通路1：fallback → 整体跳过
    reported = rng.random((n, J)) < recalls         # 通路2：逐表漏报
    hole = fb | (~reported).any(axis=1)
    return hole.mean()

def analytic(J, p_main=p_main, p_join=p_join, p_fb=p_fb):
    recalls = [p_main] + [p_join] * (J - 1)
    return p_fb + (1 - p_fb) * (1 - np.prod(recalls))

Js = [1, 2, 3, 4]
sim = [sim_hole_prob(J) for J in Js]
ana = [analytic(J) for J in Js]
print("单查询权限缺口概率（模拟 vs 解析）：")
print("%-8s%-16s%-16s" % ("表数J", "蒙特卡洛", "解析解"))
for J, s, a in zip(Js, sim, ana):
    print("%-8d%-16s%-16s" % (J, "%.2f%%" % (s * 100), "%.2f%%" % (a * 100)))

# 累积风险：每天 K 次问答，至少出现一次缺口的概率
Ks = np.arange(1, 101)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar([str(J) for J in Js], [s * 100 for s in sim], color="#d62728", alpha=0.8, label="模拟")
axes[0].plot([str(J) for J in Js], [a * 100 for a in ana], "ko--", label="解析")
axes[0].set_xlabel("查询涉及受保护表数 J"); axes[0].set_ylabel("单查询缺口概率 %")
axes[0].set_title("单次查询：权限缺口概率 vs join 复杂度"); axes[0].legend()
for J in (2, 3, 4):
    a = analytic(J)
    axes[1].plot(Ks, (1 - (1 - a) ** Ks) * 100, label="J=%d（单次 %.1f%%）" % (J, a * 100))
axes[1].axhline(50, color="gray", ls=":", lw=1)
axes[1].set_xlabel("每日问答次数 K"); axes[1].set_ylabel("至少一次缺口概率 %")
axes[1].set_title("累积风险：静默缺口随查询量复合")
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle("实验②：行权限 = 概率性执行（对比：AST 只读闸门恒为 0）", fontsize=12)
plt.tight_layout()
plt.savefig(OUT + "/w15d1_permission_risk.png", dpi=110, bbox_inches="tight")
plt.show()
cum3 = 1 - (1 - analytic(3)) ** Ks
k50 = int(Ks[np.searchsorted(cum3, 0.5)]) if (cum3 >= 0.5).any() else None
print("结论：J=3 时单次缺口 ≈ %.1f%%，日查询约 %s 次即过 50%% 概率至少一次静默越权读。" % (analytic(3) * 100, k50))
print("对照：check_sql_read 的 AST 只读闸门是确定性 0——同一链路里两种保证强度并存，这正是架构审查要点。")

## 实验③：确定性加固对照——表注册对账提取

加固方向：表提取不走 LLM 自报，改为**对着数据源表注册清单扫 SQL 文本**（生产级实现即 sqlglot AST 遍历——
LnkChatBI 的 `check_sql_read` 已依赖 sqlglot，工具是现成的）。本实验用 mallcre 真实表名 + 词边界正则演示同一思想：
召回恒 100%，与问题无关、可复现；对照模拟 LLM 自报（实验②参数）的随机召回。

In [ ]:
import re
# mallcre 真实表注册子集（数据源同步后 CoreTable 即这份清单）
REGISTRY = ["bi_d_position", "bi_b_tenant", "bi_b_contract", "bi_d_store",
            "bi_cre_m3position", "bi_cre_m3contract", "bi_d_floor"]

def registry_extract(sql):
    """确定性表提取：对注册清单做词边界扫描（AST 遍历的平替演示）。"""
    return sorted({t for t in REGISTRY if re.search(r"\b" + re.escape(t) + r"\b", sql, re.IGNORECASE)})

def write_op_detector(sql):
    """只读闸门演示：写操作黑名单（sqlglot AST 类型黑名单的平替）。"""
    first = re.match(r"\s*(\w+)", sql.strip().lower())
    return first.group(1) in {"insert", "update", "delete", "create", "drop", "alter", "merge", "grant", "truncate"}

sqls = {
    "Q1 铺位身份路径": "SELECT STORE_NAME,BUILDING_NAME,FLOOR_NAME,POSITION_CODE,POSITION_STATE FROM bi_d_position WHERE POSITION_CODE = 'A101' LIMIT 1000",
    "Q2 铺位+租户链": "SELECT p.POSITION_CODE, t.TENANT_NAME FROM bi_d_position p JOIN bi_b_tenant t ON p.CONT_NO = t.TENANT_CODE WHERE p.FLOOR_NAME = 'L2 二层' LIMIT 1000",
    "W1 写操作(应拦)": "UPDATE bi_d_position SET POSITION_STATE = 2 WHERE POSITION_CODE = 'A101'",
    "W2 写操作(应拦)": "DELETE FROM bi_b_tenant WHERE TENANT_CODE = 'TEN_DEMO_001'",
}
print("%-18s%-36s%s" % ("SQL", "注册提取", "只读?"))
for name, s in sqls.items():
    print("%-18s%-36s%s" % (name, ",".join(registry_extract(s)) or "(无)", "PASS" if not write_op_detector(s) else "BLOCK"))

# 召回对照：1000 次模拟自报 vs 注册扫描
n_trials = 1000
recall_sim = []
for _ in range(n_trials):
    reported = (rng.random() < p_main) + (rng.random() < p_join)  # 主表 + 1 join 表
    recall_sim.append(reported / 2.0)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(recall_sim, bins=[0, 0.25, 0.75, 1.01], color="#d62728", alpha=0.75, label="LLM 自报（模拟 %d 次）" % n_trials)
ax.axvline(1.0, color="#2ca02c", lw=3, label="注册对账扫描（恒 100%）")
ax.set_xticks([0, 0.5, 1.0]); ax.set_xlabel("表提取召回率"); ax.set_ylabel("频次")
ax.set_title("实验③：表提取召回——概率性自报 vs 确定性注册对账")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT + "/w15d1_recall_compare.png", dpi=110, bbox_inches="tight")
plt.show()
rs = np.array(recall_sim)
print("结论：自报召回均值 %.1f%%（%.1f%% 的查询存在漏报）；注册对账恒 100%% 且零随机性。" % (rs.mean() * 100, (rs < 1.0).mean() * 100))
print("加固成本注记：sqlglot 已是 LnkChatBI 依赖（check_sql_read 在用），表提取改 AST 遍历不引新依赖。")

## 实验④：L1-L3 验收判定器 v0.1（D6 Demo 的机器靶子）

把验收口径写成可执行判分器。判据全部落在 chat record log 的字段上（`sql` / `tables` / `sql-data` / `terminologies` 块 / 答案文本），
四条样例轨迹用 mallcre 种子数据实测素材构造（A101 → `LOC_DEMO_L202` 空置快闪铺：state=2、CONT_NO='-'、END_DATE=2099）：

| 轨迹 | 设计意图 |
|---|---|
| T1 | 全过：身份路径 + join 链 + 状态引用 + 术语条件注入且答案引用 |
| T2 | L1 过、L2 挂：只查铺位不 join（说不出租户/合同链） |
| T3 | L1 挂：只回状态字段串，无 POSITION_CODE 谓词、无层级列（PT-W4 反例原样翻译） |
| T4 | L1/L2 过、L3 挂：规则条件没进术语块，答案只说"不符合"不引条件 |

In [ ]:
HIER = ["STORE_NAME", "BUILDING_NAME", "FLOOR_NAME", "POSITION_CODE"]
ALIAS = {"A101": "LOC_DEMO_L202"}   # D3 term-aliases 将生成的别名映射（种子码风格）

def norm_sql(sql):
    return re.sub(r"\s+", " ", sql).strip().lower()

def grade_l1(traj):
    s = norm_sql(traj["sql"])
    pred_ok = ("position_code = 'a101'" in s) or ("position_code = 'loc_demo_l202'" in s)
    cols_ok = all(c.lower() in s for c in HIER)
    return pred_ok and cols_ok

def grade_l2(traj):
    s = norm_sql(traj["sql"])
    join_ok = ("join" in s) and any(t in s for t in ("bi_b_tenant", "bi_b_contract"))
    state_ok = ("position_state" in s) or ("end_date" in s)
    return join_ok and state_ok

def grade_l3(traj):
    terms = " ".join(traj.get("terminologies", [])).lower()
    cond_ok = ("空置" in terms) and (("position_state" in terms) or ("end_date" in terms))
    cited = any(k in traj["answer_text"].lower() for k in ("空置", "2099", "end_date", "position_state"))
    return cond_ok and cited

trajectories = {
    "T1 全过": dict(
        sql="SELECT STORE_NAME, BUILDING_NAME, FLOOR_NAME, POSITION_CODE, POSITION_STATE, END_DATE, TENANT_NAME FROM bi_d_position p LEFT JOIN bi_b_tenant t ON p.CONT_NO = t.TENANT_CODE WHERE POSITION_CODE = 'A101' LIMIT 1000",
        terminologies=["A101: 铺位别名→LOC_DEMO_L202；空置判定=POSITION_STATE=2 或 END_DATE>=当前且 CONT_NO='-'"],
        answer_text="A101（L2-02 快闪铺）当前为空置状态：POSITION_STATE=2，无关联租约（CONT_NO='-'）。"),
    "T2 L2挂": dict(
        sql="SELECT STORE_NAME, BUILDING_NAME, FLOOR_NAME, POSITION_CODE, POSITION_STATE FROM bi_d_position WHERE POSITION_CODE = 'A101' LIMIT 1000",
        terminologies=["A101: 铺位别名→LOC_DEMO_L202"],
        answer_text="A101 状态是 2。"),
    "T3 L1挂": dict(
        sql="SELECT POSITION_STATE FROM bi_d_position WHERE FLOOR_NAME = 'L2 二层' LIMIT 1000",
        terminologies=[],
        answer_text="状态字段值是 2，不能出租。"),
    "T4 L3挂": dict(
        sql="SELECT STORE_NAME, BUILDING_NAME, FLOOR_NAME, POSITION_CODE, POSITION_STATE FROM bi_d_position p LEFT JOIN bi_b_tenant t ON p.CONT_NO = t.TENANT_CODE WHERE POSITION_CODE = 'A101' LIMIT 1000",
        terminologies=["A101: 铺位别名→LOC_DEMO_L202"],
        answer_text="A101 不符合出租条件。"),
}

print("%-10s%-10s%-10s%-10s%s" % ("轨迹", "L1(必达)", "L2(必达)", "L3(必达)", "Demo 门槛"))
score = {}
for name, t in trajectories.items():
    g = {"L1": grade_l1(t), "L2": grade_l2(t), "L3": grade_l3(t)}
    score[name] = g
    gate = "达标 ✓" if all(g.values()) else "不达标 ✗"
    print("%-10s%-10s%-10s%-10s%s" % (name, "PASS" if g["L1"] else "FAIL", "PASS" if g["L2"] else "FAIL", "PASS" if g["L3"] else "FAIL", gate))

fig, ax = plt.subplots(figsize=(7.2, 3.2))
mat = np.array([[1 if score[n][l] else 0 for l in ("L1", "L2", "L3")] for n in trajectories])
ax.imshow(mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(3), ["L1 语义理解\n(身份路径)", "L2 业务链\n(join+状态)", "L3 规则判断\n(条件注入+引用)"])
ax.set_yticks(range(len(trajectories)), list(trajectories.keys()))
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, "PASS" if mat[i, j] else "FAIL", ha="center", va="center", fontsize=10,
                color="black" if mat[i, j] else "white", fontweight="bold")
ax.set_title("实验④：L1-L3 验收判定器 v0.1（D6 Demo 通过线 = 三绿）")
plt.tight_layout()
plt.savefig(OUT + "/w15d1_l1l3_grader.png", dpi=110, bbox_inches="tight")
plt.show()
print("结论：验收口径不必等人读答案——判定器直接吃 record log。T3 即 PT-W4 反例（字段代替身份）的 LnkChatBI 翻译，判分器稳定识别。")

## 总结：今天四个实验合起来说明的一件事

NL→SQL 组装链路里，**确定性组件（SSE 契约不变量、AST 只读闸门、注册对账提取、L1-L3 判分器）与概率性组件（LLM 生成、表自报、权限合并）是分层的**。
架构审查的任务不是消灭概率性组件（那是 LLM 的本职），而是**把安全与验收性质全部下沉到确定性层**。

- 实验①③④ 是确定性层的样板：契约、提取、验收都可机器复核；
- 实验② 量化了当前还留在概率层的部分（行权限），并给出零新依赖的加固路径；
- 这套「确定性判定器 + 概率性生成器」的组合，就是 W15-D6 Demo 的验收底座，也是 Semantic Model 未来所有消费方的通用护栏形状。